# STAR Door-State Classifier - Google Colab Training

Fine-tunes **YOLOv8n-cls** on the **DeepDoors2** dataset (gasparramoa/DeepDoors2) to classify a rectified camera frame as `open`, `closed`, or `ajar`.
Outputs `door_state_yolov8n.onnx` ready to drop into `final_capstone/compliance-engine/star_compliance/models/`.

**Dataset:** 3000 RGB images, 1000 per class (closed / open / semi-open), 480x640, from https://github.com/gasparramoa/DeepDoors2

**Runtime:** Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 is fine, ~15-20 min end to end).

**Prereq:** Add a shortcut to the DeepDoors 2 folder (https://drive.google.com/drive/folders/1SxVKeJ9RBcoJXHSHw-LWaLGG07BZT-b5) into your personal My Drive before running section 4.

**Sections:**
1. Verify GPU
2. Install dependencies
3. Paths and hyperparameters
4. Mount Drive and locate the DeepDoors2 dataset
5. Normalize dataset layout (`Semi-open` -> `ajar`, drop `Depth/`, use `RGB/`)
6. Train YOLOv8n-cls
7. Validate on held-out test split
8. Export to ONNX (opset 12, simplified)
9. md5 + size report
10. Download the ONNX file to your local machine

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install --quiet \
    'ultralytics>=8.3.0' \
    'onnx>=1.16' \
    'onnxruntime>=1.17' \
    'onnxsim>=0.4.35' \
    'opencv-python>=4.9'

## 3. Paths and hyperparameters

Tweak `EPOCHS`, `IMG_SIZE`, `BATCH_SIZE` here if you want to experiment.

In [ ]:
import os

WORK_DIR = '/content/door_state_training'
DATASET_RAW = f'{WORK_DIR}/dataset-raw'
DATASET_ROOT = f'{WORK_DIR}/dataset'
RUNS_DIR = f'{WORK_DIR}/runs'
OUTPUT_ONNX = f'{WORK_DIR}/door_state_yolov8n.onnx'

EPOCHS = 50
IMG_SIZE = 320
BATCH_SIZE = 64
DEVICE = 0

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
%cd {WORK_DIR}

## 4. Mount Drive and locate the DeepDoors2 dataset

**One-time setup (do this in a browser first):**
1. Open https://drive.google.com/drive/folders/1SxVKeJ9RBcoJXHSHw-LWaLGG07BZT-b5
2. Right-click the `DeepDoors 2` folder -> **Organize -> Add shortcut** -> anywhere under your My Drive

The cell below mounts Drive and recursively searches MyDrive for a folder whose name matches DeepDoors2 (with or without a space), then symlinks it in as `dataset-raw/`. No file copying - faster and saves Colab disk.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

candidates = []
for p in Path('/content/drive/MyDrive').rglob('*'):
    if not p.is_dir():
        continue
    name = p.name.lower().replace(' ', '').replace('-', '').replace('_', '')
    if name in {'deepdoors2', 'deepdoors'}:
        candidates.append(p)

if not candidates:
    raise RuntimeError(
        'Could not find a DeepDoors 2 folder under /content/drive/MyDrive. '
        'Add a shortcut to it from '
        'https://drive.google.com/drive/folders/1SxVKeJ9RBcoJXHSHw-LWaLGG07BZT-b5 '
        'into your My Drive first.'
    )

print('Found candidate DeepDoors folders in MyDrive:')
for c in candidates:
    print(f'  {c}')

source = candidates[0]
print(f'\nUsing: {source}')

if os.path.islink(DATASET_RAW):
    os.remove(DATASET_RAW)
elif os.path.isdir(DATASET_RAW):
    import shutil as _sh
    _sh.rmtree(DATASET_RAW)

os.symlink(str(source), DATASET_RAW)
print(f'Linked: {DATASET_RAW} -> {source}')

## 5. Normalize dataset layout

YOLOv8 classification expects:

```
dataset/
  train/{open,closed,ajar}/*.jpg
  val/{open,closed,ajar}/*.jpg
  test/{open,closed,ajar}/*.jpg
```

DeepDoors2 ships with both `RGB/` and `Depth/` modalities under `Door Classification/`. We keep only `RGB/` and rename `Semi-open` -> `ajar`.

Source layout after download: `.../Door Classification/RGB/{Test,Train,Val}/{Closed,Open,Semi-open}/*.jpg`

In [ ]:
import shutil
from pathlib import Path

rgb_root = None
for p in Path(DATASET_RAW).rglob('RGB'):
    if p.is_dir() and any((p / s).is_dir() for s in ('Train', 'Val', 'Test')):
        rgb_root = p
        break

if rgb_root is None:
    raise RuntimeError('RGB/ folder with Train/Val/Test subdirs not found under '
                       f'{DATASET_RAW}. Re-check the download step.')
print(f'Found RGB root at: {rgb_root}')

split_map = {'Train': 'train', 'Val': 'val', 'Test': 'test'}
class_map = {'Closed': 'closed', 'Open': 'open', 'Semi-open': 'ajar'}

for split_src, split_dst in split_map.items():
    for cls_src, cls_dst in class_map.items():
        src_dir = rgb_root / split_src / cls_src
        dst_dir = Path(DATASET_ROOT) / split_dst / cls_dst
        dst_dir.mkdir(parents=True, exist_ok=True)
        if not src_dir.exists():
            print(f'  skip (missing): {src_dir}')
            continue
        for f in src_dir.iterdir():
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                shutil.copy2(f, dst_dir / f.name)

print('\nImages per split/class:')
for split in ['train', 'val', 'test']:
    for cls in ['open', 'closed', 'ajar']:
        count = sum(1 for _ in (Path(DATASET_ROOT) / split / cls).glob('*'))
        print(f'  {split:<5} / {cls:<7}  {count:4d} images')

## 6. Train YOLOv8n-cls

Metrics to watch in the per-epoch output:

| Metric | Target |
|---|---|
| top1 accuracy | >= 0.85 |
| top1 (open) | >= 0.90 |
| top1 (closed) | >= 0.90 |
| top1 (ajar) | >= 0.80 |

DeepDoors2 is well balanced (1000 per class), so ajar should train much better than on the smaller DoorDetect-Class-Dataset.

In [ ]:
!yolo classify train \
    model=yolov8n-cls.pt \
    data={DATASET_ROOT} \
    epochs={EPOCHS} \
    imgsz={IMG_SIZE} \
    batch={BATCH_SIZE} \
    device={DEVICE} \
    patience=15 \
    project={RUNS_DIR} \
    name=door_state_yolov8n \
    exist_ok=true \
    save_period=10

## 7. Validate on held-out test split

Acceptance thresholds for the capstone:
- Overall top-1 accuracy: >= 0.85
- Per-class top-1: open >= 0.90, closed >= 0.90, ajar >= 0.80

In [ ]:
BEST_PT = f'{RUNS_DIR}/door_state_yolov8n/weights/best.pt'
assert os.path.isfile(BEST_PT), f'Expected best.pt at {BEST_PT}'

!yolo classify val \
    model={BEST_PT} \
    data={DATASET_ROOT} \
    imgsz={IMG_SIZE} \
    split=test \
    device={DEVICE} \
    project={RUNS_DIR} \
    name=door_state_eval \
    exist_ok=true

## 8. Export to ONNX

Opset 12 with simplification; matches what `door_state_classifier.py` expects at runtime.

In [ ]:
!yolo export \
    model={BEST_PT} \
    format=onnx \
    imgsz={IMG_SIZE} \
    opset=12 \
    simplify=true

BEST_ONNX = f'{RUNS_DIR}/door_state_yolov8n/weights/best.onnx'
assert os.path.isfile(BEST_ONNX), f'ONNX export failed; expected {BEST_ONNX}'

shutil.copy2(BEST_ONNX, OUTPUT_ONNX)
print(f'Copied: {OUTPUT_ONNX}')

## 9. md5 + size report

Record these in `final_capstone/compliance-engine/star_compliance/models/README.md` alongside the trained weights.

In [ ]:
import hashlib

def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

size_bytes = os.path.getsize(OUTPUT_ONNX)
digest = md5_of(OUTPUT_ONNX)
print(f'ONNX:  {OUTPUT_ONNX}')
print(f'Size:  {size_bytes} bytes')
print(f'md5:   {digest}')

## 10. Download ONNX to your local machine

Drop the downloaded file into
`final_capstone/compliance-engine/star_compliance/models/door_state_yolov8n.onnx`
in the STAR repo, update `models/README.md` with the md5 + dataset citation
(DeepDoors2, gasparramoa), run `pytest tests/` in the compliance-engine
package, then commit and push.

In [ ]:
from google.colab import files
files.download(OUTPUT_ONNX)